# Micro-Project 1: Party Demographic Composition, 1998–2024
### ANES Time Series Cumulative Data File (1948–2024), February 5, 2026 release
#### Janna Hoyne

**Problem statement.** This project measures how and when the demographic composition of U.S. party identifiers changed between 1998 and 2024, and distinguishes change within the parties from change in the underlying population.

**Original hypothesis (H1).** Compositional change occurred in bursts rather than steadily.
- shift magnitudes differ substantially across intervals rather than being roughly constant
- on at least one dimension, the parties diverge from each other rather than both tracking the national trend

**Scope.** This project terminates at Step 2 (Prepare) of the data science
process. Deliverables are a documented, analysis-ready dataset and
descriptive findings — no models or significance testing.

**Citation.** American National Election Studies. 2026. *ANES Time Series
Cumulative Data File* [dataset and documentation]. February 5, 2026 version.
www.electionstudies.org

**Analysis window.** 1998–2024, covering nine ANES survey years: 1998, 2000, 2002, 2004, 2008, 2012, 2016, 2020, 2024. Change is measured across the eight adjacent intervals below. ANES discontinued midterm Time Series studies after 2002, so the first three intervals span two years and the remaining five span four; per-interval change is divided by interval length to make them comparable.

- 1998 - 2000
- 2000 - 2002
- 2002 - 2004
- 2004 - 2008
- 2008 - 2012
- 2012 - 2016
- 2016 - 2020
- 2020 - 2024

## Acquire

### Environment

Only NumPy, pandas, and the Python standard library are used. Library versions are printed so results can be reproduced against the same environment.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 120)

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.3.3 | numpy 2.3.5


### Load pre-cleaned subset

The full cumulative file is 73,745 rows × 1,030 columns (~3.6 GB parsed). A pre-cleaning step reduced it to 53 variables and saved anes_subset.csv

- Blanks are encoded as a single space, and one space in a column of digits forces pandas to type the entire column as object — 1,017 of 1,030 columns on first load. 
- na_values=[' '] at parse time resolves it. Notably, the 13 columns that did parse as numeric were exactly those with no blanks in any year, making the dtype pattern an early coverage signal.

In [2]:
work = pd.read_csv("anes_subset.csv")

print("Loaded:", work.shape)

Loaded: (73745, 53)


### Scope and structural drops

**Why 1998?** 1998 is the earliest election year at which the demographic coding schemes used here are stable, and it precedes the major national events of the following two decades. Earlier years require substantially more recoding to compare.

**Why only nine years across a 26-year span?** ANES discontinued midterm Time Series studies after 2002, so the series runs biennially through 2004 and every four years thereafter.

Three geography variables were also dropped: VCF0901a (state FIPS) covers
only 3,105 of 5,521 cases in 2024, VCF0901b (state abbreviation) is entirely
blank in 2024, and VCF0113 (South/nonsouth) has no 2024 data. VCF0112
(Census Region) has full coverage in all nine years and is retained.

In [3]:
KEEP = [
    # structure
    'VERSION', 'VCF0004', 'VCF0006', 'VCF0006a',  # release version; year of study; case ID; cross-year ID
    'VCF0009z', 'VCF0010z', 'VCF0011z', 'VCF9999',  # full-sample weights (1970 types 0/1/2); post-election weight
    'VCF0013', 'VCF0014', 'VCF0017', 'VCF0016', 'VCF1016',  # completion post/pre; mode; cross vs panel; days post-IW

    # party identification
    'VCF0301', 'VCF0302', # Party Id(PID) scalet; PID initial response;
    'VCF0303', 'VCF0305',  # PID 3-cat summary; PID strength

    # registration / turnout / vote
    'VCF0703', 'VCF0702', 'VCF0704', 'VCF0707', 'VCF0713',  # reg+turnout summary; voted; pres vote; House vote; pre-election intent
    'VCF0701', 'VCF0737', 'VCF0738', 'VCF0704a',  # registered pre; registered post; party of registration; pres vote by party

    # demographics
    'VCF0101', 'VCF0102', 'VCF0103', 'VCF0104',  # age; age group; birth cohort; gender
    'VCF0105a', 'VCF0105b', 'VCF0106', 'VCF0107', # race-ethnicity 7-cat; 4-cat; race 3-cat; Hispanic origin type
    'VCF0110', 'VCF0140', 'VCF0140a', 'VCF0114',  # # education 4-cat; 6-cat; 7-cat; family income group
    'VCF0112', 'VCF0127', 'VCF0146', 'VCF0147', 'VCF0113',  # census region; union HH; home ownership; marital status; South/nonsouth

    # -religion
    'VCF0128', 'VCF0130', 'VCF0130a', 'VCF0846',  # religion major group; attendance 1970-; attendance 1990-; religion important

    # economic situation
    'VCF0880', 'VCF0880a', 'VCF0870', 'VCF0156',  # better/worse off; how much; economy past year; laid off last 6 mo

    # geography
    'VCF0901a', 'VCF0901b',  # state FIPS; state postal abbreviation

    #engagement
    'VCF0310',  # interest in the elections
]

print(f"{len(KEEP)} variables | duplicates: {len(KEEP) - len(set(KEEP))}")

54 variables | duplicates: 0


## Prepare

### Missing-value recoding

ANES encodes missingness inside the numeric codes, not as blanks. A party ID of 0 means "refused"; an education value of 9 means "NA, refused, or no pre-election interview." These are ordinary numbers to pandas, so an unrecoded mean of VCF0140a would treat 9 as higher than 7 — ranking refusals above an advanced degree.

This differs from the structural blanks handled at parse time. Those mean the file holds no value; these mean a real response was recorded as non-substantive. The codes vary by variable, so each was transcribed from the codebook into `MISSING_CODES` with its meaning documented inline.

In [4]:
modern = work[work['VCF0004'] >= 1998].copy()
modern = modern.drop(columns=['VCF0701', 'VCF0738'])
modern = modern.drop(columns=['VCF0901a', 'VCF0901b', 'VCF0113'])

In [5]:
# Missing codes differ per variable (variable codebook), so no blanket rule.
MISSING_CODES = {
    'VCF0703':  [0],       # DK/NA vote or registration; no Post IW
    'VCF0301':  [0],       # NA/RF party ID; no Pre IW
    'VCF0303':  [0],       # DK/NA/other/refused
    'VCF0105b': [0, 9],    # 0 = pre-1966; 9 = DK/REF/NA
    'VCF0140a': [8, 9],    # 8 = DK; 9 = NA/RF/no Pre IW
}

for col, codes in MISSING_CODES.items():
    modern[col] = modern[col].replace(codes, np.nan)

print(modern[list(MISSING_CODES)].isna().sum())

VCF0703     2850
VCF0301      248
VCF0303      248
VCF0105b     316
VCF0140a     601
dtype: int64


### Derived variables
Five variables built from the cleaned columns:

- **`race_measure flags`** a measurement break: race was interviewer-observed through 1998 and self-reported from 2000, and the change lands on the window's first boundary. No recoding can fix this, so the flag keeps the discontinuity in the data rather than only in a footnote.

- **`educ4`** collapses the 7-point scale to 4 categories. Not simplification for readability — it repairs two instrument changes. Category 4 ("12 grades plus non-academic training") exists only from 2012, and 2020 merged categories 1 and 2 at source. Folding 3+4 and 1+2 makes all nine years comparable.

- **`college`** is a binary BA-or-higher indicator, immune to both changes above since every affected category falls below the BA line.

- **`nonwhite`** collapses codes 2, 3, 4 (Black, Hispanic, Other/multiple) against code 1 (White non-Hispanic). The 4-category variable is used rather than the 7-category VCF0105a because the finer categories are thin and unstable across years.

- **`registered`** treats codes 2 and 3 as registered. Per the codebook, all respondents reporting a vote are coded 3 regardless of whether they reported being registered, so registration is partly inferred — compounding the self-report inflation noted in the limitations.

The three binary indicators use an .isna() guard. Without it, missing values evaluate as False and would silently become "not college," "white," or "not registered" — 601, 316, and 2,850 cases respectively.

In [6]:
# derived variables

# Race measurement changed from interviewer observation (through 1998) to
# respondent self-report (2000+). Flag keeps the break visible.
modern['race_measure'] = np.where(
    modern['VCF0004'] <= 1998, 'observed', 'self-report'
)

# Education collapsed 7 -> 4.
# Repairs two instrument changes.
#   - Cat 4 ("12 grades + non-academic training") exists only 2012+;
#     folding 3+4 makes pre- and post-2012 comparable.
#   - 2020 merged cats 1 and 2 at source; folding 1+2 matches that.
ed_map = {1: 1, 2: 1, 3: 2, 4: 2, 5: 3, 6: 4, 7: 4}
modern['educ4'] = modern['VCF0140a'].map(ed_map)

# Binary BA-or-higher. .isna() guard stops missing values from silently
# becoming "not college".
modern['college'] = np.where(
    modern['VCF0140a'].isna(), np.nan,
    (modern['VCF0140a'] >= 6).astype(float)
)

# Non-white indicator (codes 2=Black, 3=Hispanic, 4=Other/multiple).
modern['nonwhite'] = np.where(
    modern['VCF0105b'].isna(), np.nan,
    (modern['VCF0105b'] != 1).astype(float)
)

# Registered = code 2 (registered, didn't vote) or 3 (voted, so registered).
modern['registered'] = np.where(
    modern['VCF0703'].isna(), np.nan,
    modern['VCF0703'].isin([2, 3]).astype(float)
)

print(modern[['race_measure', 'educ4', 'college', 'nonwhite', 'registered']].notna().sum())

race_measure    32118
educ4           31517
college         31517
nonwhite        31802
registered      29268
dtype: int64


In [7]:
print("Shape:", modern.shape)
print("Derived:", [c for c in modern.columns if not c.startswith('VCF')])

Shape: (32118, 53)
Derived: ['race_measure', 'educ4', 'college', 'nonwhite', 'registered']


### First look: unweighted composition

**Direction of the measure.** These tables ask what share of Democrats hold a BA — the denominator is the party. The reverse measure, attachment, asks what share of BA-holders are Democrats. Both are defensible; they answer different questions, and this project measures composition throughout.

The first two tables are unweighted, retained deliberately to demonstrate why weighting is not optional

In [8]:
# COMPOSITION: within each party, what share holds a BA or higher?
# Denominator is the party, not the education group.
comp = modern[['VCF0004', 'VCF0303', 'college']].dropna()

college_by_party = (
    comp.groupby(['VCF0004', 'VCF0303'])['college']
        .mean().unstack().mul(100).round(1)
)
college_by_party.columns = ['Democrat', 'Independent', 'Republican']

print("Share with BA or higher, by party (%), unweighted\n")
print(college_by_party.to_string())

Share with BA or higher, by party (%), unweighted

         Democrat  Independent  Republican
VCF0004                                   
1998         26.0         17.9        33.5
2000         27.7         21.0        38.3
2002         31.5         29.0        40.7
2004         30.6         15.7        33.1
2008         20.1         11.4        28.9
2012         28.9         25.0        37.5
2016         42.2         24.5        39.7
2020         51.8         32.4        40.5
2024         54.3         27.4        39.2


In [9]:
# COMPOSITION: within each party, what share is non-white?
comp2 = modern[['VCF0004', 'VCF0303', 'nonwhite']].dropna()

nonwhite_by_party = (
    comp2.groupby(['VCF0004', 'VCF0303'])['nonwhite']
         .mean().unstack().mul(100).round(1)
)
nonwhite_by_party.columns = ['Democrat', 'Independent', 'Republican']

print("Non-white share, by party (%), unweighted\n")
print(nonwhite_by_party.to_string())
print("\nNOTE: 1998 = interviewer-observed race; 2000+ = self-reported.")

Non-white share, by party (%), unweighted

         Democrat  Independent  Republican
VCF0004                                   
1998         34.4         22.2        11.4
2000         33.4         22.7        13.6
2002         30.9         17.2        11.6
2004         38.8         36.2        14.8
2008         63.3         49.6        23.1
2012         55.2         38.2        17.9
2016         38.8         34.7        14.3
2020         36.1         36.8        14.2
2024         35.7         40.5        16.8

NOTE: 1998 = interviewer-observed race; 2000+ = self-reported.


The problem. Unweighted non-white share among Democrats: 34–39% through 2004 → 63.3% in 2008 → 55.2% in 2012 → 38.8% in 2016.

- **Not real demographic change.** Population shifts are gradual and one-directional. A 25-point rise and 25-point fall in eight years is not one.
- **Not sample size.** These are percentages, not counts. A larger sample is more precise, not larger in value.
- **Education breaks in the same year.** College share fell for all three parties in 2008, then recovered — though national attainment has never declined. Two unrelated variables breaking simultaneously implies one cause: sample composition.
- **Weighting removes it.** 2008 drops from 63.3% to 38.2% and the full series becomes coherent.

In [10]:
def wmean(df, val, wt='VCF0009z'):
    """Weighted mean: sum(w*x) / sum(w). groupby().mean() cannot weight."""
    return np.average(df[val], weights=df[wt])

In [11]:
sub_ed = modern[['VCF0004', 'VCF0303', 'college', 'VCF0009z']].dropna()

college_wtd = (
    sub_ed.groupby(['VCF0004', 'VCF0303'])
          .apply(wmean, 'college', include_groups=False)
          .unstack().mul(100).round(1)
)
college_wtd.columns = ['Democrat', 'Independent', 'Republican']

print("Share with BA or higher, by party (%), WEIGHTED\n")
print(college_wtd.to_string())

Share with BA or higher, by party (%), WEIGHTED

         Democrat  Independent  Republican
VCF0004                                   
1998         21.2         13.3        27.1
2000         20.8         14.7        31.0
2002         20.9         17.7        30.3
2004         26.6         11.2        28.5
2008         26.9         13.3        34.9
2012         28.6         21.2        33.6
2016         34.0         18.2        32.5
2020         42.0         26.4        33.6
2024         43.7         18.5        27.5


### National benchmarks

- A rising college share within a party does not by itself indicate realignment — national attainment rose throughout this period, so both parties would gain graduates regardless. Party-level movement is therefore read as the difference from the national line: a party can rise in absolute terms while falling relative to the population.
- The same logic applies to age. Both parties' mean age rises as the population ages; only the gap between them is informative.

In [12]:
# Overall college share by year - the benchmark both parties.
overall = modern[['VCF0004', 'college', 'VCF0009z']].dropna()

overall_wtd = (
    overall.groupby('VCF0004')
           .apply(wmean, 'college', include_groups=False)
           .mul(100).round(1)
)

print("Overall college share (%), weighted\n")
print(overall_wtd.to_string())

Overall college share (%), weighted

VCF0004
1998    22.5
2000    23.7
2002    24.3
2004    25.6
2008    28.0
2012    29.4
2016    31.0
2020    36.7
2024    34.8


### Group sizes

Percentages are only as stable as the cells behind them. Independents number 95 in 2002 and 121 in 2004. Small enough that sampling error accounts for most of the visible variation in that column. Independent estimates are reported but not read as evidence of change. Partisan cells are adequate in every year (smallest: 468).

In [13]:
print(pd.crosstab(modern['VCF0004'], modern['VCF0303']).to_string())

VCF0303   1.0  2.0   3.0
VCF0004                 
1998      656  145   468
2000      888  224   680
2002      703   95   669
2004      591  121   483
2008     1365  264   653
2012     3103  792  1995
2016     1939  579  1729
2020     3836  968  3441
2024     2644  380  2459


### Validity check: does interview mode confound 2024?

- 2024 reintroduced face-to-face interviewing, absent in 2020. Mode would bias the partisan comparison only if two conditions both hold: mode is associated with education, and mode composition differs by party.
- Weighted college share across 2024 modes ranges from 22.3% (all personal, n=433) to 48.0% (personal pre / video post, n=339) — a spread wider than the entire partisan gap being reported. Modes with fewer than 20 cases are disregarded.

In [14]:
# Does 2024 college share differ by interview mode?
m24 = modern[modern['VCF0004'] == 2024][['VCF0017', 'VCF0303', 'college', 'VCF0009z']].dropna()

print("2024 N by mode:")
print(m24['VCF0017'].value_counts().sort_index().to_string())

print("\n2024 college share by mode (weighted):")
print(m24.groupby('VCF0017').apply(wmean, 'college', include_groups=False).mul(100).round(1).to_string())

2024 N by mode:
VCF0017
0     433
1       3
3      51
4    4154
6     339
7     169
8      16
9       3

2024 college share by mode (weighted):
VCF0017
0     22.3
1     19.0
3     35.8
4     35.2
6     48.0
7     32.6
8     53.5
9    100.0


The second condition does not hold. 
- Mode composition is nearly identical across parties: 79.5% of Democrats and 80.8% of Republicans were interviewed by internet, with face-to-face shares of 8.3% and 8.6%. The largest imbalance (mode 6, 7.7% vs 5.9%) is too small to move the gap by more than a fraction of a point.

**The 2024 partisan comparison is not confounded by mode.**
- Mode does explain the apparent national decline in college share from 36.7% (2020) to 34.8% (2024): attainment did not fall, the sample composition changed.

- **Principle applied throughout:** report gaps, not levels. Any bias affecting a whole year's sample — mode, oversampling, differential response — hits both parties roughly equally and cancels out of the difference between them. Levels are reported for context; the difference between parties is the measure.

In [15]:
# Does mode composition differ by party in 2024?
# Row-normalized: what % of each party came through each mode
mode_by_party = pd.crosstab(
    m24['VCF0303'], m24['VCF0017'], normalize='index'
).mul(100).round(1)
mode_by_party.index = ['Democrat', 'Independent', 'Republican']

print("2024 mode composition by party (%)\n")
print(mode_by_party.to_string())

2024 mode composition by party (%)

VCF0017        0    1    3     4    6    7    8    9
Democrat     8.3  0.1  1.0  79.5  7.7  2.9  0.4  0.1
Independent  7.1  0.0  0.9  84.0  2.9  4.9  0.3  0.0
Republican   8.6  0.0  1.0  80.8  5.9  3.4  0.2  0.0


In [16]:
sub_reg = modern[['VCF0004', 'VCF0303', 'registered', 'VCF0009z']].dropna()

reg_wtd = (
    sub_reg.groupby(['VCF0004', 'VCF0303'])
           .apply(wmean, 'registered', include_groups=False)
           .unstack().mul(100).round(1)
)
reg_wtd.columns = ['Democrat', 'Independent', 'Republican']

print("Self-reported registration rate (%), WEIGHTED\n")
print(reg_wtd.to_string())

Self-reported registration rate (%), WEIGHTED

         Democrat  Independent  Republican
VCF0004                                   
1998         83.0         66.9        82.4
2000         87.7         64.0        87.6
2002         86.3         67.3        91.2
2004         90.8         72.3        93.0
2008         87.7         65.3        91.6
2012         93.5         77.0        93.2
2016         92.0         71.7        93.0
2020         96.0         81.0        95.6
2024         96.7         72.3        94.8


### Age and gender

Value counts are inspected before recoding to confirm the codebook's listed
codes match what is actually present, and to detect anything undocumented.

In [17]:
for col in ['VCF0104', 'VCF0102', 'VCF0103']:
    print(f"\n{col}")
    print(modern[col].value_counts(dropna=False).sort_index().to_string())


VCF0104
VCF0104
0      154
1    14733
2    17220
3       11

VCF0102
VCF0102
0     889
1    2194
2    4865
3    5558
4    5384
5    5691
6    4713
7    2824

VCF0103
VCF0103
0     768
1    2466
2    7335
3    8695
4    9100
5    3116
6     612
7      26


In [18]:
print(modern[modern['VCF0104'] == 3].groupby('VCF0004').size())

VCF0004
2016    11
dtype: int64


In [19]:
MISSING_CODES.update({
    'VCF0104': [0],   # NA; no Pre IW
    'VCF0102': [0],   # NA; DK; RF; no Pre IW
    'VCF0103': [0],   # NA; DK; RF; no Pre IW
})

for col in ['VCF0104', 'VCF0102', 'VCF0103']:
    modern[col] = modern[col].replace(MISSING_CODES[col], np.nan)

print(modern[['VCF0104', 'VCF0102', 'VCF0103']].isna().sum())

VCF0104    154
VCF0102    889
VCF0103    768
dtype: int64


Gender carries the same kind of measurement break as race, in a different year: interviewer-coded through 2012, self-described from 2016. Category 3 ("Other") was introduced at the same time, consistent with the change in method.

Only 11 respondents are coded 3, all in 2016, though the codebook lists 2020 and 2024 sources. The category is excluded from the binary female measure because 11 cases cannot support a stable estimate. Those respondents remain in the dataset — excluded from one measure, not deleted.

In [20]:
# ---- DERIVED: GENDER ----

# Gender was interviewer-coded through 2012 and self-described from 2016,
# when category 3 ("Other") was also introduced. Same kind of instrument
# change as race, so it gets the same kind of flag.
modern['gender_measure'] = np.where(
    modern['VCF0004'] <= 2012, 'observed', 'self-report'
)

# Binary female indicator. Code 3 (n=11) is set to NaN: the category only
# exists from 2016 and has too few cases for stable estimates. Excluded
# from this measure, not from the dataset.
modern['female'] = np.where(
    modern['VCF0104'].isin([1, 2]),
    (modern['VCF0104'] == 2).astype(float),
    np.nan
)

print(modern['female'].value_counts(dropna=False))

female
1.0    17220
0.0    14733
NaN      165
Name: count, dtype: int64


In [21]:
# Gender composition of each party
sub_g = modern[['VCF0004', 'VCF0303', 'female', 'VCF0009z']].dropna()

female_wtd = (
    sub_g.groupby(['VCF0004', 'VCF0303'])
         .apply(wmean, 'female', include_groups=False)
         .unstack().mul(100).round(1)
)
female_wtd.columns = ['Democrat', 'Independent', 'Republican']

print("Female share, by party (%), WEIGHTED\n")
print(female_wtd.to_string())

Female share, by party (%), WEIGHTED

         Democrat  Independent  Republican
VCF0004                                   
1998         56.2         56.8        50.0
2000         59.9         54.4        51.8
2002         57.6         56.5        52.6
2004         55.5         50.7        46.8
2008         58.6         47.3        51.2
2012         55.0         52.2        48.4
2016         55.7         49.3        48.3
2020         56.5         48.2        47.6
2024         55.9         52.3        46.3


**Averaging an ordinal variable.** Mean age group assumes equal spacing between categories. Categories 2–6 are 10-year bands, but category 1 spans 8 years and category 7 spans 25, so the mean compresses movement at the top end. Adequate for detecting a trend; share-in-group measures (under-35, 65+) would avoid the assumption if age became central.

In [22]:
# Age: mean age group by party (1 = 17-24 - 7 = 75+)
sub_a = modern[['VCF0004', 'VCF0303', 'VCF0102', 'VCF0009z']].dropna()

age_wtd = (
    sub_a.groupby(['VCF0004', 'VCF0303'])
         .apply(wmean, 'VCF0102', include_groups=False)
         .unstack().round(2)
)
age_wtd.columns = ['Democrat', 'Independent', 'Republican']

print("Mean age group, by party, WEIGHTED (1=17-24 ... 7=75+)\n")
print(age_wtd.to_string())

Mean age group, by party, WEIGHTED (1=17-24 ... 7=75+)

         Democrat  Independent  Republican
VCF0004                                   
1998         3.71         3.33        3.38
2000         3.69         3.04        3.62
2002         3.53         3.31        3.52
2004         3.62         3.79        3.77
2008         3.66         3.35        3.91
2012         3.73         3.49        3.92
2016         3.69         3.45        4.00
2020         3.77         3.46        4.15
2024         3.90         3.47        3.98


In [23]:
# National mean age group - the benchmark both parties
overall_age = modern[['VCF0004', 'VCF0102', 'VCF0009z']].dropna()

age_national = (
    overall_age.groupby('VCF0004')
               .apply(wmean, 'VCF0102', include_groups=False)
               .round(2)
)

print("National mean age group, weighted\n")
print(age_national.to_string())

National mean age group, weighted

VCF0004
1998    3.54
2000    3.58
2002    3.49
2004    3.69
2008    3.72
2012    3.77
2016    3.78
2020    3.89
2024    3.91


In [24]:
overall_f = modern[['VCF0004', 'female', 'VCF0009z']].dropna()

f_national = (
    overall_f.groupby('VCF0004')
             .apply(wmean, 'female', include_groups=False)
             .mul(100).round(1)
)

print("National female share (%), weighted\n")
print(f_national.to_string())

National female share (%), weighted

VCF0004
1998    54.2
2000    56.1
2002    55.6
2004    51.5
2008    54.9
2012    52.1
2016    51.9
2020    51.8
2024    51.6


### Attrition in the registration variable

VCF0703 is missing for 2,850 cases (8.9%), against 0.8% for party ID. The codebook lists "no Post IW" among the missing reasons, and the mechanism is structural: party ID comes from the pre-election wave and registration from the post-election wave, so anyone who dropped out between them has one but not the other.

Missingness is therefore non-random, and its direction determines whether the registration findings are overstated or understated.

In [25]:
# Is missingness on VCF0703 explained by post-election attrition?
chk = modern[['VCF0004', 'VCF0303', 'VCF0703', 'VCF0013']].copy()
chk['reg_missing'] = chk['VCF0703'].isna()

print("VCF0703 missing, by post-election completion:")
print(pd.crosstab(chk['VCF0013'], chk['reg_missing'], normalize='index').mul(100).round(1))

VCF0703 missing, by post-election completion:
reg_missing  False  True 
VCF0013                  
0             13.4   86.6
1             99.7    0.3


Independents drop out at higher rates than partisans in nearly every year (14.7% vs 8.6% for Democrats in 2024). Assuming dropouts are disproportionately unregistered — plausible, since survey attrition and civic disengagement tend to correlate, though untestable here by definition — observed registration rates are inflated, and inflated more for independents.

**The bias therefore works against the finding.** As a worst-case bound: if every missing 2024 independent were unregistered, their rate would fall from 72.3% to roughly 62%, while Democrats would fall from 96.7% to roughly 88% - widening the gap from 24 to 27 points. The reported independent deficit is conservative under any assumption about the missing cases.

Democrat–Republican differences in attrition are small and change sign across years, so partisan comparisons are unaffected.

In [26]:
sub_att = chk[['VCF0004', 'VCF0303', 'reg_missing']].dropna(subset=['VCF0303'])

att_by_party = (
    sub_att.groupby(['VCF0004', 'VCF0303'])['reg_missing']
           .mean().unstack().mul(100).round(1)
)
att_by_party.columns = ['Democrat', 'Independent', 'Republican']

print("\nVCF0703 missing rate (%), by party\n")
print(att_by_party.to_string())


VCF0703 missing rate (%), by party

         Democrat  Independent  Republican
VCF0004                                   
1998          0.6          1.4         0.6
2000         15.8         16.5        11.5
2002         12.7         15.8        10.2
2004         14.4         15.7         8.7
2008         10.4         11.0         7.8
2012          1.1          3.4         1.0
2016         12.9         17.4        13.9
2020          8.9          9.6        10.1
2024          8.6         14.7        10.6


### Missingness profile

Missing rates per variable per year after all recodes. Three patterns, plus one exception:

**Structural 100% cells** are not data quality problems. Presidential vote variables are empty in the midterm years 1998 and 2002 because no presidential election occurred. VCF0737 ends after 2008, VCF0016 after 2016, VCF1016 after 2020, and VCF0114 was not fielded in 2002.

**A 4.4% block in 2024.** Seven otherwise-complete variables (VCF0302, VCF0147, VCF0128, VCF0880, VCF0880a, VCF0870, VCF0310) show exactly 4.4% missing in 2024 and 0.0% elsewhere — 245 of 5,521 respondents. An identical rate across unrelated items suggests a questionnaire-version or administration difference affecting a specific group rather than item-level non-response.

**Education missingness rises in 2024** to 5.9%, above the 4.4% block. This is the year with the largest reported partisan gap, so the elevated rate is noted alongside the mode limitation.

**Exception:** VCF0156 (laid off in last 6 months) is 45.7% missing in 2012 and 100% in 2002 — a split-sample administration rather than either pattern above. The variable is not used in the analysis.

In [27]:
# Missing rate (%) per variable per year, after all recodes.
miss = (
    modern.isna()
          .groupby(modern['VCF0004'])
          .mean()
          .mul(100).round(1)
          .T
)
miss.columns = miss.columns.astype(int)

# Show only variables with any missingness
print(miss[miss.sum(axis=1) > 0].to_string())

VCF0004      1998  2000   2002  2004  2008   2012   2016   2020   2024
VCF9999     100.0   0.0    0.0   0.0   0.0  100.0    0.0   10.0   10.1
VCF0016       0.0   0.0    0.0   0.0   0.0    0.0    0.0  100.0  100.0
VCF1016       0.0   0.0    0.0   0.0   0.0    0.0    0.0    0.0  100.0
VCF0301       0.9   0.8    2.9   1.4   1.7    0.4    0.5    0.4    0.7
VCF0302       0.0   0.0    0.0   0.0   0.0    0.0    0.0    0.0    4.4
VCF0303       0.9   0.8    2.9   1.4   1.7    0.4    0.5    0.4    0.7
VCF0703       0.8  14.3   11.6  12.2   9.8    1.4   14.0    9.6   10.1
VCF0704     100.0   0.0  100.0   0.0   0.0    0.0    0.0    0.0    0.0
VCF0713     100.0   0.0  100.0   0.0   0.0    0.0    0.0    0.0    0.0
VCF0737       0.0   0.0    0.0   0.0   0.0  100.0  100.0  100.0  100.0
VCF0704a    100.0   0.0  100.0   0.0   0.0    0.0    0.0    0.0  100.0
VCF0102       1.2   0.5    0.7   0.0   1.9    1.0    2.8    4.2    5.1
VCF0103       1.2   0.5    0.7   0.0   1.9    1.0    2.8    4.3    2.8
VCF010

In [28]:
modern.to_csv("anes_analysis.csv", index=False)
print("Saved:", modern.shape)

Saved: (32118, 55)


### Assessing H1

**H1(a): supported.** Per-year change in Democratic college share ranges
from −0.20 to +2.85 percentage points across the eight intervals — change
is not steady. The largest movements fall in 2002→2004 and 2016→2020,
though the former rests on the smallest sample in the series (591 Democrats
in 2004) and is not strong evidence on its own.

**H1(b): supported on education, not on race.** Against the national college
benchmark, Democrats moved from −1.3 to +8.9 and Republicans from +4.6 to
−7.3 — divergence in opposite directions. On race, both parties diversified
at roughly the national rate, and the partisan gap was 23.3 points in 1998
against 24.1 in 2024.

Because the survey runs at two- to four-year intervals, no individual
national event can be isolated as a cause. The intervals in which change
concentrated are reported; attributing them is beyond what this data
supports.

In [29]:
# H1(a): rate of change per interval.
# Divided by interval length so two-year and four-year gaps are comparable.
years = college_wtd.index.to_series().diff()
per_year = college_wtd.diff().div(years, axis=0).round(2).dropna()

print("College share: change per year, by interval (pct pts/yr)\n")
print(per_year.to_string())

College share: change per year, by interval (pct pts/yr)

         Democrat  Independent  Republican
VCF0004                                   
2000        -0.20         0.70        1.95
2002         0.05         1.50       -0.35
2004         2.85        -3.25       -0.90
2008         0.07         0.53        1.60
2012         0.43         1.97       -0.32
2016         1.35        -0.75       -0.28
2020         2.00         2.05        0.28
2024         0.43        -1.97       -1.53


### Save analysis file

**This file is partially cleaned, by design.** Only the eight variables in MISSING_CODES have had their missing codes converted to NaN. The remaining 28 VCF columns retain variable-specific **`missing codes`** as live numeric values and must be recoded against the codebook before use.

The contrast between VCF0105a (0.1% missing in 2024) and VCF0105b (1.2%) shows the risk: the same underlying question, but only b was recoded, so a still counts its 9s as valid responses.

The CSV carries no metadata, so cleaning state cannot be inferred from the file itself. The four-way listing below is printed into the notebook as the record of which columns are safe to use.

In [30]:
# Save
STRUCTURAL = ['VCF0004', 'VCF0006', 'VCF0006a', 'VCF0009z', 'VCF0010z',
              'VCF0011z', 'VCF9999', 'VCF0013', 'VCF0014', 'VCF0017',
              'VCF0016', 'VCF1016']

CLEANED = list(MISSING_CODES.keys())
DERIVED = [c for c in modern.columns if not c.startswith('VCF')]
NOT_CLEANED = [c for c in modern.columns
               if c not in CLEANED + DERIVED + STRUCTURAL]

print("CLEANED (missing codes recoded):")
print("  ", CLEANED)
print("\nDERIVED (built from cleaned variables):")
print("  ", DERIVED)
print("\nSTRUCTURAL (IDs, weights, flags - no substantive missing codes):")
print("  ", STRUCTURAL)
print("\nNOT CLEANED (raw missing codes still present):")
print("  ", NOT_CLEANED)

modern.to_csv("anes_analysis.csv", index=False)
print("\nSaved:", modern.shape)

CLEANED (missing codes recoded):
   ['VCF0703', 'VCF0301', 'VCF0303', 'VCF0105b', 'VCF0140a', 'VCF0104', 'VCF0102', 'VCF0103']

DERIVED (built from cleaned variables):
   ['race_measure', 'educ4', 'college', 'nonwhite', 'registered', 'gender_measure', 'female']

STRUCTURAL (IDs, weights, flags - no substantive missing codes):
   ['VCF0004', 'VCF0006', 'VCF0006a', 'VCF0009z', 'VCF0010z', 'VCF0011z', 'VCF9999', 'VCF0013', 'VCF0014', 'VCF0017', 'VCF0016', 'VCF1016']

NOT CLEANED (raw missing codes still present):
   ['VCF0302', 'VCF0305', 'VCF0702', 'VCF0704', 'VCF0707', 'VCF0713', 'VCF0737', 'VCF0704a', 'VCF0101', 'VCF0105a', 'VCF0106', 'VCF0107', 'VCF0110', 'VCF0140', 'VCF0114', 'VCF0112', 'VCF0127', 'VCF0146', 'VCF0147', 'VCF0128', 'VCF0130', 'VCF0130a', 'VCF0846', 'VCF0880', 'VCF0880a', 'VCF0870', 'VCF0156', 'VCF0310']

Saved: (32118, 55)


In [31]:
# ---- PRE-CLEANING (run once; produced anes_subset.csv) ----
# Commented out so the notebook runs from the saved subset. Retained to
# document how the subset was produced.
#
# PATH = "anes_timeseries_cdf_csv_20260205.csv"
#
# KEEP = [
#     'VCF0004','VCF0006','VCF0006a','VCF0009z','VCF0010z','VCF0011z',
#     'VCF9999','VCF0013','VCF0014','VCF0017','VCF0016','VCF1016',
#     'VCF0301','VCF0302','VCF0303','VCF0305',
#     'VCF0703','VCF0702','VCF0704','VCF0707','VCF0713','VCF0701',
#     'VCF0737','VCF0738','VCF0704a',
#     'VCF0101','VCF0102','VCF0103','VCF0104','VCF0105a','VCF0105b',
#     'VCF0106','VCF0107','VCF0110','VCF0140','VCF0140a','VCF0114',
#     'VCF0112','VCF0127','VCF0146','VCF0147','VCF0113',
#     'VCF0128','VCF0130','VCF0130a','VCF0846',
#     'VCF0880','VCF0880a','VCF0870','VCF0156',
#     'VCF0901a','VCF0901b','VCF0310',
# ]
#
# raw = pd.read_csv(PATH, low_memory=False)          # 73,745 x 1,030
# present = [v for v in KEEP if v in raw.columns]    # 'VERSION' not in CSV
# work = raw[present].copy()
#
# text_cols = ['VCF0901b']
# num_cols = [c for c in work.columns if c not in text_cols]
# work[num_cols] = work[num_cols].apply(pd.to_numeric, errors='coerce')
#
# work.to_csv("anes_subset.csv", index=False)
#Janna Hoyne